# Python Interview Challenge Bank

**Goal:** the ~25 Python coding challenges that show up most repeatedly across real interview prep sources — not a random grab-bag. Each one was cross-checked against multiple independent sources before being included (see `INTERVIEW_CHALLENGES_CONTEXT.md` for the exact sources and methodology).

**Format:** same reading-first philosophy as the rest of this project — problem statement (phrased the way it's commonly asked), a short explanation of the approach and why it's asked, and a real, executed, verified solution with test cases printed. No TODO placeholders — read it, run it, then try covering the code and re-deriving it yourself as practice.

**How to actually practice with this:** the first pass, read + run each cell to see the pattern. The second pass, collapse the code cell, re-read only the problem statement, and try writing your own solution from scratch before looking. That's what actually builds interview fluency — recognizing "oh, this is a two-pointer problem" or "this needs a hash map," not memorizing the exact code.

---


## A. Strings

### A1. Reverse a string

**Asked as:** "Write a function that reverses a string." Sometimes with a follow-up: "now do it without using slicing or built-ins."

**What it tests:** basic string/list manipulation, and whether you know strings are immutable in Python (so "reversing in place" really means building a new string).

**Approach:** Python's slice `[::-1]` does it in one line — know this, but also know how to do it manually (two-pointer swap via a list, since strings can't be mutated in place) in case that's explicitly requested.


In [1]:
def reverse_string_pythonic(s):
    return s[::-1]

def reverse_string_manual(s):
    chars = list(s)                    # strings are immutable -- work on a list instead
    left, right = 0, len(chars) - 1
    while left < right:
        chars[left], chars[right] = chars[right], chars[left]
        left += 1
        right -= 1
    return ''.join(chars)

for test in ['hello', 'Python', '', 'a']:
    assert reverse_string_pythonic(test) == reverse_string_manual(test) == test[::-1]
    print(f'{test!r:10} -> {reverse_string_pythonic(test)!r}')


'hello'    -> 'olleh'
'Python'   -> 'nohtyP'
''         -> ''
'a'        -> 'a'


### A2. Check if a string is a palindrome

**Asked as:** "Determine if a given string reads the same forwards and backwards." Often with a twist: ignore case, spaces, and punctuation (e.g. "A man, a plan, a canal: Panama" should count as a palindrome).

**What it tests:** string manipulation, and whether you handle the "clean the input first" edge case instead of only handling the trivial version.

**Approach:** normalize the string (lowercase, strip non-alphanumeric characters), then compare it to its own reverse.


In [2]:
def is_palindrome(s):
    cleaned = ''.join(ch.lower() for ch in s if ch.isalnum())   # strip spaces/punctuation, ignore case
    return cleaned == cleaned[::-1]

tests = ['racecar', 'hello', 'A man, a plan, a canal: Panama', 'Was it a car or a cat I saw?', '']
for t in tests:
    print(f'{t!r:40} -> {is_palindrome(t)}')


'racecar'                                -> True
'hello'                                  -> False
'A man, a plan, a canal: Panama'         -> True
'Was it a car or a cat I saw?'           -> True
''                                       -> True


### A3. Check if two strings are anagrams

**Asked as:** "Given two strings, determine if one is an anagram of the other" — or the harder variant, "group a list of strings into anagram clusters."

**What it tests:** whether you reach for sorting (simple, O(n log n)) or a character-count hash map (O(n), the better answer when asked to optimize).

**Approach:** two strings are anagrams if they contain exactly the same characters with the same frequency — `sorted(s1) == sorted(s2)` is the one-liner; `Counter` is the O(n) version and also naturally solves the "group anagrams" variant by using the sorted-letters as a dict key.


In [3]:
from collections import Counter, defaultdict

def is_anagram(s1, s2):
    return Counter(s1.replace(' ', '').lower()) == Counter(s2.replace(' ', '').lower())

print(is_anagram('listen', 'silent'))
print(is_anagram('hello', 'world'))
print(is_anagram('Dormitory', 'Dirty Room'))

def group_anagrams(words):
    groups = defaultdict(list)
    for word in words:
        key = ''.join(sorted(word))     # anagrams share the same sorted-letters key
        groups[key].append(word)
    return list(groups.values())

print(group_anagrams(['eat', 'tea', 'tan', 'ate', 'nat', 'bat']))


True
False
True
[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


### A4. Find the first non-repeating character

**Asked as:** "Given a string, return the first character that doesn't repeat." Classic example: `'swiss'` -> `'w'`.

**What it tests:** whether you use a hash map to count first (one pass) instead of nested loops (O(n²) — checking every character against every other character).

**Approach:** count every character's frequency in one pass, then walk the string again (to preserve order) and return the first one with a count of 1.


In [4]:
from collections import Counter

def first_non_repeating(s):
    counts = Counter(s)
    for ch in s:                 # walk in ORIGINAL order, not dict order, to find the FIRST one
        if counts[ch] == 1:
            return ch
    return None

for test in ['swiss', 'aabbcc', 'teeter', 'x']:
    print(f'{test!r:10} -> {first_non_repeating(test)!r}')


'swiss'    -> 'w'
'aabbcc'   -> None
'teeter'   -> 'r'
'x'        -> 'x'


### A5. Count character frequency

**Asked as:** "Given a string, count how many times each character appears" — often as a building block for a bigger problem (word frequency counters, log analysis).

**What it tests:** knowing `collections.Counter` exists (versus hand-rolling a dict with `.get(key, 0) + 1`), and being able to do both.


In [5]:
from collections import Counter

def char_frequency_manual(s):
    freq = {}
    for ch in s:
        freq[ch] = freq.get(ch, 0) + 1     # .get() with a default avoids a KeyError on first sight
    return freq

def char_frequency_counter(s):
    return dict(Counter(s))

text = 'mississippi'
print(char_frequency_manual(text))
print(char_frequency_counter(text))
assert char_frequency_manual(text) == char_frequency_counter(text)


{'m': 1, 'i': 4, 's': 4, 'p': 2}
{'m': 1, 'i': 4, 's': 4, 'p': 2}


### A6. Reverse the words in a sentence, without reversing the words themselves

**Asked as:** given `"the sky is blue"`, return `"blue is sky the"` — the WORD order reverses, but each word stays spelled forwards. A common trap is applying `[::-1]` to the whole string, which reverses the letters too.

**What it tests:** whether you split into words first before reversing, rather than reversing everything at once.


In [6]:
def reverse_words(sentence):
    words = sentence.split()          # split() also collapses multiple spaces, handling that edge case
    return ' '.join(reversed(words))

print(reverse_words('the sky is blue'))
print(reverse_words('  hello   world  '))   # extra whitespace -- split() handles it cleanly


blue is sky the
world hello


### A7. Check for balanced parentheses/brackets

**Asked as:** "Given a string of brackets `()[]{}`, determine if every opening bracket has a matching closing bracket in the correct order." Example: `'([{}])'` is balanced, `'([)]'` is not.

**What it tests:** whether you recognize this as a **stack** problem — the most-recently-opened bracket must be the next one closed (LIFO), which is exactly what a stack models.


In [7]:
def is_balanced(s):
    pairs = {')': '(', ']': '[', '}': '{'}
    stack = []
    for ch in s:
        if ch in '([{':
            stack.append(ch)
        elif ch in ')]}':
            if not stack or stack.pop() != pairs[ch]:   # nothing to match, or the wrong bracket type
                return False
    return len(stack) == 0        # any unclosed opening brackets left over? -> not balanced

for test in ['([{}])', '([)]', '((()', '', '{[()()]}']:
    print(f'{test!r:10} -> {is_balanced(test)}')


'([{}])'   -> True
'([)]'     -> False
'((()'     -> False
''         -> True
'{[()()]}' -> True


### A8. String compression (run-length encoding)

**Asked as:** compress `'aaabbc'` into `'a3b2c1'` (or the "no digit for count-of-1" variant: `'a3b2c'`). If the compressed version isn't shorter, some variants ask you to return the original string instead.

**What it tests:** iterating with awareness of the *previous* element (tracking run boundaries), a pattern that generalizes to a lot of "consecutive group" problems.


In [8]:
def compress_string(s):
    if not s:
        return s
    result = []
    count = 1
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            count += 1
        else:
            result.append(s[i - 1] + str(count))   # flush the run we just finished
            count = 1
    result.append(s[-1] + str(count))               # flush the final run
    compressed = ''.join(result)
    return compressed if len(compressed) < len(s) else s   # only compress if it's actually shorter

for test in ['aaabbc', 'abcdef', 'aabbccdd', 'aaaaaaaaaa']:
    print(f'{test!r:15} -> {compress_string(test)!r}')


'aaabbc'        -> 'aaabbc'
'abcdef'        -> 'abcdef'
'aabbccdd'      -> 'aabbccdd'
'aaaaaaaaaa'    -> 'a10'


## B. Arrays and Lists

### B1. FizzBuzz

**Asked as:** "Print numbers 1 to N. For multiples of 3, print 'Fizz'; for multiples of 5, print 'Buzz'; for multiples of both, print 'FizzBuzz'."

**What it tests:** almost nothing technical — it's a filter for "can this person write basic control flow at all." The trap: checking 3 and 5 individually with `if`/`elif` in the wrong order produces the wrong output for multiples of 15 (never reaches the combined case if `elif` order is wrong, or double-prints if using separate `if`s incorrectly).


In [9]:
def fizzbuzz(n):
    result = []
    for i in range(1, n + 1):
        if i % 15 == 0:            # check the MOST specific condition (both 3 and 5) FIRST
            result.append('FizzBuzz')
        elif i % 3 == 0:
            result.append('Fizz')
        elif i % 5 == 0:
            result.append('Buzz')
        else:
            result.append(str(i))
    return result

print(fizzbuzz(20))


['1', '2', 'Fizz', '4', 'Buzz', 'Fizz', '7', '8', 'Fizz', 'Buzz', '11', 'Fizz', '13', '14', 'FizzBuzz', '16', '17', 'Fizz', '19', 'Buzz']


### B2. Two Sum

**Asked as:** "Given an array of integers and a target, return the indices of the two numbers that add up to the target." Almost certainly the single most-asked coding interview question across all languages, not just Python.

**What it tests:** whether you jump straight to the O(n²) brute-force (check every pair) or recognize the O(n) hash-map trick: for each number, check whether `target - number` was already seen.


In [10]:
def two_sum_brute_force(nums, target):
    for i in range(len(nums)):                 # O(n^2) -- works, but not the answer that impresses
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
    return None

def two_sum_optimal(nums, target):
    seen = {}                                    # value -> index, built incrementally
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:                   # O(1) lookup instead of a nested loop
            return [seen[complement], i]
        seen[num] = i
    return None

nums, target = [2, 7, 11, 15], 9
print(two_sum_brute_force(nums, target))
print(two_sum_optimal(nums, target))
assert two_sum_brute_force(nums, target) == two_sum_optimal(nums, target)


[0, 1]
[0, 1]


### B3. Find duplicates in an array

**Asked as:** "Given a list, return the elements that appear more than once."

**What it tests:** using a `set` to track "seen" items in one pass (O(n)) instead of comparing every pair (O(n²)).


In [11]:
def find_duplicates(items):
    seen = set()
    duplicates = set()
    for item in items:
        if item in seen:
            duplicates.add(item)
        else:
            seen.add(item)
    return list(duplicates)

print(sorted(find_duplicates([1, 2, 3, 2, 4, 5, 1, 6])))
print(find_duplicates(['a', 'b', 'c']))    # no duplicates -> empty list


[1, 2]
[]


### B4. Remove duplicates from a list, preserving order

**Asked as:** "Given a list, remove duplicates but keep the original order of first appearance." The trap: `list(set(items))` removes duplicates but **does not** preserve order (sets are unordered).

**What it tests:** knowing that sets don't preserve order, and using a seen-set alongside a result list instead.


In [12]:
def remove_duplicates_preserve_order(items):
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

original = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3]
print(remove_duplicates_preserve_order(original))    # order of first appearance preserved
print(list(dict.fromkeys(original)))                  # a shorter idiom -- dicts preserve insertion order too


[3, 1, 4, 5, 9, 2, 6]
[3, 1, 4, 5, 9, 2, 6]


### B5. Find the missing number

**Asked as:** "Given an array containing n distinct numbers from 0 to n, find the one that's missing."

**What it tests:** the "sum trick" — the expected sum of 0..n has a closed-form formula, so subtracting the actual sum reveals the gap in O(n) time and O(1) extra space, no sorting or hash set needed.


In [13]:
def find_missing_number(nums):
    n = len(nums)
    expected_sum = n * (n + 1) // 2      # sum of 0..n, via the closed-form Gauss formula
    actual_sum = sum(nums)
    return expected_sum - actual_sum

print(find_missing_number([3, 0, 1]))           # n=3, missing 2
print(find_missing_number([0, 1, 2, 4, 5]))     # n=5, missing 3


2
3


### B6. Find max/min in an array without built-ins

**Asked as:** "Find the largest number in a list without using `max()`." Tests whether you understand the mechanics `max()` hides — sometimes it's the actual point of the question (checking you know the algorithm, not just the function name).


In [14]:
def find_max_manual(nums):
    if not nums:
        raise ValueError('empty list has no maximum')
    current_max = nums[0]
    for n in nums[1:]:
        if n > current_max:
            current_max = n
    return current_max

nums = [3, 7, 2, 9, 4, 1]
print(find_max_manual(nums))
assert find_max_manual(nums) == max(nums)


9


### B7. Move all zeroes to the end, in place

**Asked as:** given `[0, 1, 0, 3, 12]`, rearrange it to `[1, 3, 12, 0, 0]`, keeping the relative order of the non-zero elements, modifying the array in place (not building a new one).

**What it tests:** the "two-pointer" / "write pointer" technique — a very common pattern family in array problems (also used for removing elements, partitioning).


In [15]:
def move_zeroes(nums):
    write_pos = 0                         # tracks where the next non-zero value should go
    for read_pos in range(len(nums)):
        if nums[read_pos] != 0:
            nums[write_pos], nums[read_pos] = nums[read_pos], nums[write_pos]
            write_pos += 1
    return nums

print(move_zeroes([0, 1, 0, 3, 12]))
print(move_zeroes([0, 0, 1]))
print(move_zeroes([4, 2, 4, 0, 0, 3, 0, 5, 1, 0]))


[1, 3, 12, 0, 0]
[1, 0, 0]
[4, 2, 4, 3, 5, 1, 0, 0, 0, 0]


### B8. Merge two sorted lists

**Asked as:** "Given two sorted lists, merge them into one sorted list" — the core subroutine behind merge sort (see D3 below), so it's worth knowing on its own too.

**What it tests:** the two-pointer merge technique, O(n+m) instead of concatenating and re-sorting (which would be O((n+m) log(n+m)) — correct, but not the efficient answer).


In [16]:
def merge_sorted_lists(a, b):
    result = []
    i = j = 0
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            result.append(a[i])
            i += 1
        else:
            result.append(b[j])
            j += 1
    result.extend(a[i:])     # append whatever's left in whichever list wasn't exhausted
    result.extend(b[j:])
    return result

print(merge_sorted_lists([1, 3, 5, 7], [2, 4, 6, 8]))
print(merge_sorted_lists([1, 2, 3], []))


[1, 2, 3, 4, 5, 6, 7, 8]
[1, 2, 3]


### B9. Flatten a nested list

**Asked as:** turn `[1, [2, 3, [4, 5]], 6]` into `[1, 2, 3, 4, 5, 6]`, for an arbitrary (unknown) depth of nesting.

**What it tests:** recursion — "arbitrary depth" is the signal that a fixed number of loops won't work; you need a function that calls itself when it encounters another list.


In [17]:
def flatten(nested):
    result = []
    for item in nested:
        if isinstance(item, list):
            result.extend(flatten(item))   # recurse into the nested list
        else:
            result.append(item)
    return result

print(flatten([1, [2, 3, [4, 5]], 6]))
print(flatten([1, [2, [3, [4, [5]]]]]))     # deeply nested -- recursion handles any depth


[1, 2, 3, 4, 5, 6]
[1, 2, 3, 4, 5]


## C. Numbers and Math

### C1. Fibonacci sequence: iterative, recursive, and memoized

**Asked as:** "Write a function that returns the nth Fibonacci number" — almost always followed by "what's the time complexity?" and "can you make it faster?"

**What it tests:** whether you know naive recursion is exponential (O(2ⁿ), recomputes the same values repeatedly) and can fix it with memoization (caching) or by just going iterative — a very common "here's a slow correct answer, now optimize it" interview arc.


In [18]:
def fib_recursive(n):
    if n <= 1:
        return n
    return fib_recursive(n - 1) + fib_recursive(n - 2)   # correct, but O(2^n) -- recomputes constantly

def fib_iterative(n):
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b                                              # O(n) time, O(1) space

from functools import lru_cache
@lru_cache(maxsize=None)
def fib_memoized(n):
    if n <= 1:
        return n
    return fib_memoized(n - 1) + fib_memoized(n - 2)      # O(n) time -- cached, never recomputes

for n in [0, 1, 5, 10, 20]:
    r, i, m = fib_recursive(n), fib_iterative(n), fib_memoized(n)
    assert r == i == m
    print(f'fib({n}) = {i}')


fib(0) = 0
fib(1) = 1
fib(5) = 5
fib(10) = 55
fib(20) = 6765


### C2. Factorial, recursively

**Asked as:** "Write a function to compute n! (n factorial)" — often the first recursion question asked, since the base case and recursive case are both obvious.

**What it tests:** whether you correctly define the base case (`0! = 1`) to stop the recursion, and whether you'd catch a negative input.


In [19]:
def factorial(n):
    if n < 0:
        raise ValueError('factorial is undefined for negative numbers')
    if n == 0:
        return 1                          # base case -- without this, the recursion never stops
    return n * factorial(n - 1)

for n in [0, 1, 5, 10]:
    print(f'{n}! = {factorial(n)}')

try:
    factorial(-1)
except ValueError as e:
    print('ValueError:', e)


0! = 1
1! = 1
5! = 120
10! = 3628800
ValueError: factorial is undefined for negative numbers


### C3. Check if a number is prime, and generate primes up to N

**Asked as:** "Write a function to check if a number is prime" and/or "print all primes up to N" (the second part is the Sieve of Eratosthenes).

**What it tests:** for the single-number check, whether you only test divisors up to `sqrt(n)` (any factor larger than that has a corresponding factor smaller than it, already checked) instead of looping all the way to n. For the "primes up to N" version, whether you know the Sieve exists at all — it's dramatically faster than checking each number individually.


In [20]:
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):   # only need to check up to sqrt(n), not all the way to n
        if n % i == 0:
            return False
    return True

print([n for n in range(2, 30) if is_prime(n)])

def sieve_of_eratosthenes(limit):
    is_prime_arr = [True] * (limit + 1)
    is_prime_arr[0] = is_prime_arr[1] = False
    for i in range(2, int(limit ** 0.5) + 1):
        if is_prime_arr[i]:
            for multiple in range(i * i, limit + 1, i):   # cross off every multiple of i at once
                is_prime_arr[multiple] = False
    return [n for n, prime in enumerate(is_prime_arr) if prime]

print(sieve_of_eratosthenes(50))


[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]


### C4. Check if a number is a palindrome

**Asked as:** "Determine if an integer reads the same forwards and backwards" — often with a follow-up: "can you do it without converting to a string?"

**What it tests:** the string-based version is trivial; the numeric version (repeatedly extracting digits with `% 10` and `// 10` to build the reverse) tests whether you can manipulate numbers directly.


In [21]:
def is_number_palindrome_string(n):
    s = str(n)
    return s == s[::-1]

def is_number_palindrome_numeric(n):
    if n < 0:
        return False              # negative numbers can never be palindromes (the '-' breaks symmetry)
    original, reversed_num = n, 0
    while n > 0:
        digit = n % 10
        reversed_num = reversed_num * 10 + digit
        n //= 10
    return original == reversed_num

for test in [121, 12321, 123, -121, 0]:
    r1, r2 = is_number_palindrome_string(test), is_number_palindrome_numeric(test)
    assert r1 == r2
    print(f'{test:6} -> {r1}')


   121 -> True
 12321 -> True
   123 -> False
  -121 -> False
     0 -> True


## D. Algorithms

### D1. Binary search

**Asked as:** "Given a sorted array, find the index of a target value" — the canonical O(log n) algorithm, and a frequent building block for harder problems ("find the insertion point," "find the first/last occurrence").

**What it tests:** whether you can implement it correctly, including the classic off-by-one traps in the `low`/`high`/`mid` bounds.


In [22]:
def binary_search_iterative(arr, target):
    low, high = 0, len(arr) - 1
    while low <= high:
        mid = (low + high) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            low = mid + 1          # target is in the right half
        else:
            high = mid - 1          # target is in the left half
    return -1

def binary_search_recursive(arr, target, low=0, high=None):
    if high is None:
        high = len(arr) - 1
    if low > high:
        return -1
    mid = (low + high) // 2
    if arr[mid] == target:
        return mid
    elif arr[mid] < target:
        return binary_search_recursive(arr, target, mid + 1, high)
    else:
        return binary_search_recursive(arr, target, low, mid - 1)

sorted_arr = [1, 3, 5, 7, 9, 11, 13, 15]
for target in [7, 1, 15, 4]:
    i_result = binary_search_iterative(sorted_arr, target)
    r_result = binary_search_recursive(sorted_arr, target)
    assert i_result == r_result
    print(f'target={target:3} -> index {i_result}')


target=  7 -> index 3
target=  1 -> index 0
target= 15 -> index 7
target=  4 -> index -1


### D2. Bubble sort

**Asked as:** "Implement a sorting algorithm from scratch" — bubble sort is the simplest to reason about (repeatedly swap adjacent out-of-order pairs), even though it's rarely used in practice (O(n²)). It's asked to test whether you understand sorting mechanics, not to test if you'd actually use it.


In [23]:
def bubble_sort(arr):
    arr = arr.copy()                 # don't mutate the caller's original list
    n = len(arr)
    for i in range(n):
        swapped = False
        for j in range(n - 1 - i):    # each pass, the largest unsorted element 'bubbles' to the end
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swapped = True
        if not swapped:                # optimization: if nothing swapped, it's already sorted -- stop early
            break
    return arr

print(bubble_sort([5, 2, 9, 1, 5, 6]))
print(bubble_sort([1, 2, 3]))          # already sorted -- exits after just 1 pass thanks to the swapped flag


[1, 2, 5, 5, 6, 9]
[1, 2, 3]


### D3. Quicksort

**Asked as:** "Implement quicksort" — the more commonly requested sort in interviews (versus bubble sort) since it demonstrates the divide-and-conquer pattern and average O(n log n) performance.

**What it tests:** understanding partitioning around a pivot, and recursion on the two resulting sub-problems.


In [24]:
def quicksort(arr):
    if len(arr) <= 1:
        return arr                     # base case: 0 or 1 elements is already sorted
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quicksort(left) + middle + quicksort(right)   # recurse on each partition, then combine

print(quicksort([5, 2, 9, 1, 5, 6]))
print(quicksort([]))
print(quicksort([3, 3, 3, 1]))


[1, 2, 5, 5, 6, 9]
[]
[1, 3, 3, 3]


## E. Data Structures

### E1. Implement a Stack

**Asked as:** "Implement a stack with push, pop, peek, and is_empty" — usually to confirm you know LIFO semantics before using it to solve a real problem (like A7's balanced-brackets check above).

**What it tests:** correct LIFO behavior, and handling the empty-stack edge case for `pop()`/`peek()` instead of letting it crash unhelpfully.


In [25]:
class Stack:
    def __init__(self):
        self._items = []

    def push(self, item):
        self._items.append(item)          # add to the END of the list -- that's the 'top' of the stack

    def pop(self):
        if self.is_empty():
            raise IndexError('pop from an empty stack')
        return self._items.pop()            # removes and returns from the END -- LIFO

    def peek(self):
        if self.is_empty():
            raise IndexError('peek from an empty stack')
        return self._items[-1]

    def is_empty(self):
        return len(self._items) == 0

s = Stack()
s.push(1)
s.push(2)
s.push(3)
print(s.peek())     # 3 -- the most recently pushed item
print(s.pop())        # 3, removed
print(s.pop())        # 2, removed
print(s.is_empty())


3
3
2
False


### E2. Implement a Queue

**Asked as:** "Implement a queue with enqueue and dequeue" — tests FIFO semantics, and often a follow-up: "why is a plain list a bad choice for this?"

**What it tests:** knowing that `list.pop(0)` is O(n) (Item 103 in the Effective Python track covers this in depth) — `collections.deque` supports O(1) removal from the front, which is the efficient, correct choice.


In [26]:
from collections import deque

class Queue:
    def __init__(self):
        self._items = deque()      # deque, NOT a plain list -- O(1) from both ends, unlike list.pop(0)

    def enqueue(self, item):
        self._items.append(item)     # add to the back

    def dequeue(self):
        if self.is_empty():
            raise IndexError('dequeue from an empty queue')
        return self._items.popleft()  # remove from the FRONT -- O(1) with deque, O(n) with a plain list

    def is_empty(self):
        return len(self._items) == 0

q = Queue()
q.enqueue('a')
q.enqueue('b')
q.enqueue('c')
print(q.dequeue())   # 'a' -- first in, first out
print(q.dequeue())   # 'b'
print(q.is_empty())


a
b
False
